# Practical PyTorch · Part 6 — Companion Notebook

Why a plain `Linear` network struggles with images, and how **convolution** fixes it — building up to **LeNet**, the first famous CNN. No training, just the shapes.

## 1. Your MLP can already eat an image

An image is just a grid of numbers — a 28×28 digit is 784 of them. Flatten it and a `Linear` takes it. It runs... but flattening throws away everything about which pixel is next to which.

In [ ]:
import torch
import torch.nn as nn

mlp = nn.Sequential(
    nn.Flatten(),            # a 28x28 image -> a row of 784 numbers
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 10),      # 10 digit classes
)
img = torch.rand(1, 1, 28, 28)   # one 1-channel 28x28 image
print(mlp(img).shape)            # torch.Size([1, 10]) — it runs!

## 2. Three things go wrong

1. **It ignores layout** — once flattened, neighbouring pixels are just 'input #1' and 'input #2'.
2. **The parameters explode** — see below.
3. **It can't generalize across position** — shift the digit one pixel and every input changes.

In [ ]:
big = nn.Linear(28 * 28, 1000)
print(sum(p.numel() for p in big.parameters()))   # 785,000 — for one tiny grayscale image, one layer

## 3. Convolution: slide a small filter

`nn.Conv2d` drags a tiny window (say 5×5) across the image, looking for one pattern and reporting *where* it found it. One small filter, reused everywhere. That fixes all three at once:

- **respects layout** — it only ever looks at a pixel together with its neighbours;
- **barely any parameters** — a 5×5 filter is 25 numbers, reused at every position;
- **generalizes across position** — the same filter slides everywhere, so a pattern is found wherever it appears.

## 4. Pooling and flattening

- **`nn.MaxPool2d`** shrinks the image, keeping the strongest signal in each patch (e.g. each 2×2 block).
- **`nn.Flatten`** turns the digested feature grid into a row to hand to the final `Linear` layers.

So a CNN is a sandwich: convolutions + pooling to *see*, then `Linear` layers to *decide*.

## 5. Building LeNet (LeCun, 1998)

The first CNN to do a real job — reading handwritten digits for mail and checks. Same `nn.Sequential` you know, now with the new layers. Note the parameter count vs. the 785,000 a single flat `Linear` wanted.

In [ ]:
lenet = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2),  # slide 6 filters over a 1-channel image
    nn.ReLU(),
    nn.MaxPool2d(2),                            # halve the height and width
    nn.Conv2d(6, 16, kernel_size=5),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),                               # grid -> a flat row, to hand to Linear
    nn.Linear(16 * 5 * 5, 120),
    nn.ReLU(),
    nn.Linear(120, 84),
    nn.ReLU(),
    nn.Linear(84, 10),                          # 10 outputs — one per digit
)

total = sum(p.numel() for p in lenet.parameters())
print(f'{total:,} parameters')                  # 61,706 parameters

Convolutions to see, `Linear` layers to decide — and far fewer parameters than the flat version. (This is the untrained architecture; it runs but won't recognize a digit until trained, which is Phase II.)

**Next: Part 7 — Reading a Real One: ResNet Up Close.**